# Session 1 · Meet the Mess

**Goal:** load 100,000 mock offender records, look around, and watch dirty data give confident, wrong answers. Then fix one thing and see which answers change.

**How to use this notebook**
- Click a gray code cell and press **Shift + Enter** to run it. Go top to bottom.
- 🧪 **Your turn** cells contain a `___` blank. Replace it with your answer and run the cell; it tells you whether you're right. Answers are at the bottom if you're stuck.
- Nothing here can break anything. If things get strange, choose **Run → Clear state**, then run from the top.

> ⚠️ **Mock data only.** Everything in this course is fictional. Never load real offender data into any tool that isn't approved for criminal justice information.

In [ ]:
# ▶ SETUP: run this cell first (Shift + Enter). It finds the course files and builds today's data.
SESSION = 1

import sys
from pathlib import Path

def _find_course_files():
    for folder in [Path.cwd(), *Path.cwd().parents]:          # the course's Git folder in Databricks
        if (folder / "lnl" / "__init__.py").exists():
            return folder
    try:                                                         # found from the notebook's own location
        here = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
        for folder in Path("/Workspace" + here).parents:
            if (folder / "lnl" / "__init__.py").exists():
                return folder
    except Exception:
        pass
    raise RuntimeError("Can't find the course files. Open this notebook from inside the course's Git folder in Databricks.")

COURSE = _find_course_files()
sys.path.insert(0, str(COURSE))
from lnl import env
PATHS = env.prepare(SESSION)

## 1. Load it like a first-timer
Most tutorials load a CSV with one line and the default settings. Let's do exactly that.

In [ ]:
import pandas as pd
from lnl import check
from lnl.check import ___

df = pd.read_csv(PATHS["data"])
print(f"{len(df):,} rows x {df.shape[1]} columns")
df.head(10)

## 2. Look around
Five functions cover most first looks at a new dataset:

| Function | What it tells you |
|---|---|
| `df.shape` | how many rows and columns |
| `df.head()` | the first few rows |
| `df.info()` | each column's type and how many values are filled in |
| `df["column"].value_counts()` | every distinct value and how often it appears |
| `df.describe()` | quick statistics |

In [ ]:
df.info()

Two things to notice:

- **`tdcj_number` is `int64`**, a number, while **`sid_number` is still text**. pandas saw digits and converted one ID column on its own, dropping the leading zero (`02381457` became `2381457`), and left the other alone. Storing IDs as whole numbers is a legitimate choice, and our tables do it on purpose, but here nobody chose it and nothing was checked. A search for the ID as people write it, `"02381457"`, now finds nothing.
- **The date columns are text** (`object` or `str`). There are nine different date formats in this file, so pandas left them all as strings. You can't sort, filter, or do date math on them correctly.

In [ ]:
df[["tdcj_number", "sid_number", "date_of_birth", "sentence_date"]].head()

**`tdcj_number` is an ID.** IDs are labels: you look records up and join on them, but you never add or average them. The source writes TDCJ and SID numbers as 8-digit text with leading zeros, and our tables store them as whole numbers (BIGINT). Neither form is wrong. Letting a tool pick one for you, unchecked, is. The safe first step is to read the ID columns as text, exactly as written, so nothing changes before you've looked:

In [ ]:
ids = pd.read_csv(PATHS["data"], usecols=["tdcj_number", "sid_number"],
                  dtype={"tdcj_number": str, "sid_number": str})
ids.head(3)

Session 2 checks every ID, converts TDCJ and SID numbers to whole numbers on purpose, and makes `tdcj_number` the record's ID for everything that follows. For the rest of today we leave pandas' guess in place so you can see what it breaks.

One more: several columns are yes/no **flags stored as a single character**. This one should have exactly two values:

In [ ]:
df["protective_custody"].value_counts(dropna=False)

`Y` and `N`, but also `T` and `F` (two source systems were merged), lowercase versions, `U` for unknown, and whitespace. Session 4 sorts this out.

And the **coded** columns: unit location, race, religion, security threat group (STG) affiliation, offense code, and violence code. Each one should hold a short list of standard codes:

In [ ]:
coded = ["unit_location", "race", "religion", "stg_affiliation", "offense_code", "violence_code"]
display(pd.Series({col: df[col].nunique() for col in coded}, name="distinct values").to_frame())
df["violence_code"].value_counts(dropna=False).head(15)

There are four violence codes: NV, V1, V2 and V3. The file has dozens of spellings (`V-2`, `v2`, a bare `2`), plus codes that don't exist (`V4`). Race and religion are the same story, STG has hundreds of variations, and even units arrive as names, 3-letter codes and misspellings.

One more thing pandas did on its own: its default settings treat the text `N/A` and `None` as missing. In `stg_affiliation` those mean *no known affiliation*, a real answer, and pandas erased it for about 25,000 records without asking. Session 2 maps every coded column to agreed reference data instead.

🧪 **Your turn #1.** How many distinct values does `custody_level` have, **including blanks**? `value_counts(dropna=False)` lists every value, missing ones included.

In [ ]:
custody_counts = df["custody_level"].value_counts(dropna=False)
display(custody_counts)

answer_1 = ___      # how many rows are in the list above?  (tip: len(custody_counts))
check.s1_ex1(answer_1)

There should be five custody levels (G1 to G5). Some of those extra rows look blank. Hold that thought.

## 3. The Trust Test
Five questions a warden might reasonably ask, answered the way most people would on a first try:

In [ ]:
first_try = {
    "Q1 Records missing a custody level": df["custody_level"].isna().sum(),
    "Q2 Distinct gender values":          df["gender"].nunique(dropna=False),
    "Q3 Records with last name KENT":     (df["last_name"] == "KENT").sum(),
    "Q4 Earliest projected release date": repr(df["projected_release_date"].dropna().min()),
    "Q5 Does TDCJ #02381457 exist?":      (df["tdcj_number"].astype(str) == "02381457").any(),
}
pd.Series(first_try, name="answer").to_frame()

**Every one of those answers is wrong.** Some look obviously odd (19 genders?). Others look perfectly reasonable, like Q1, and that's worse, because nobody would question them. Let's find out why.

In [ ]:
# repr() shows invisible characters: quotes around each value, and \t for tabs
df["gender"].value_counts(dropna=False).rename(index=repr)

The file is full of values that are **only whitespace**: spaces, and sometimes tabs. To a person they look blank. To pandas, `"   "` is a perfectly good piece of text, so it isn't counted as missing, and every different amount of padding counts as a different gender.

## 4. The one fix: strip whitespace, then turn blanks into real nulls
Two steps, applied to **every** text column (you can't tell which values are blank until they've been stripped):

1. `.str.strip()` removes spaces and tabs from both ends of each value. Spaces *inside* a value, like `Gotham City`, are left alone.
2. `.mask(stripped == "")` turns what's now empty text into a real missing value, which `isna()` understands.

In [ ]:
clean = df.copy()
text_columns = clean.select_dtypes(include=["object", "string"]).columns

for col in text_columns:
    stripped = clean[col].str.strip()
    clean[col] = stripped.mask(stripped == "")

print(f"Stripped {len(text_columns)} text columns")
clean["gender"].value_counts(dropna=False)

🧪 **Your turn #2.** How many records are missing a gender now?

In [ ]:
answer_2 = ___      # use clean["gender"] and the function that counts missing values
check.s1_ex2(answer_2)

## 5. Rerun the Trust Test

In [ ]:
second_try = {
    "Q1 Records missing a custody level": clean["custody_level"].isna().sum(),
    "Q2 Distinct gender values":          clean["gender"].nunique(dropna=False),
    "Q3 Records with last name KENT":     (clean["last_name"] == "KENT").sum(),
    "Q4 Earliest projected release date": repr(clean["projected_release_date"].dropna().min()),
    "Q5 Does TDCJ #02381457 exist?":      (clean["tdcj_number"].astype(str) == "02381457").any(),
}
pd.DataFrame({"as loaded": first_try, "after strip + null": second_try})

| | Status | Why | Fixed in |
|---|---|---|---|
| Q1 | ✅ Fixed | Whitespace-only values now count as missing, which nearly doubles the count | today |
| Q2 | ✅ Fixed | M, F, and missing | today |
| Q3 | ❌ Still 0 | The file has `Kent`, `kent`, and `kEnT`, and matching is case-sensitive | Session 2 |
| Q4 | ❌ Still wrong | Dates are text, so `min()` sorts them alphabetically, not by date | Session 3 |
| Q5 | ❌ Still False | The ID column is now a number and the search is text, so they never match | Session 2 |

🧪 **Your turn #3.** Which `unit_location` has the most records?

In [ ]:
answer_3 = ___      # tip: clean["unit_location"].value_counts() lists the most common value first
check.s1_ex3(answer_3)

## Wrap-up
- Loading with defaults **silently changed the data**: pandas turned one ID column into numbers without checking it, and left dates as text.
- Whitespace-only values hide from `isna()` and split one category into many.
- Coded columns hold many spellings of each code, and pandas' defaults turned `N/A` and `None` into missing, even where they're real answers.
- One simple fix corrected two answers and left three still wrong. **An answer that looks clean isn't necessarily correct.**

**Next session:** load the file the right way, repair names and notes, handle codes that mean "unknown," map coded values to reference data, and check the keys.

---
## Answer key
```python
answer_1 = len(custody_counts)
answer_2 = clean["gender"].isna().sum()
answer_3 = clean["unit_location"].value_counts().index[0]
```